# 📘 Understanding the LLM Context Window

A practical tutorial for beginner-to-intermediate Python developers explaining what a context window is, how it is measured in tokens, how to count tokens with tiktoken, and how to detect and prevent context overflow in production code.

---

**How to use this notebook:**

1. Read each section — explanations are in the grey text cells.
2. Run the code cells with **Shift + Enter**.
3. Edit and experiment — the best way to learn is to break things.
4. Complete the **Practice Exercises** at the bottom.

> 💡 This notebook accompanies the teaching video on *Understanding the LLM Context Window*.

---

## What Is a Context Window?

A context window is the maximum amount of text an LLM can process in a single call — it is the model's entire working memory for that request. Everything the model can see is inside the context: your system prompt, the full conversation history, the new message you are sending, and the space reserved for the model's response.

### Key Points

- maximum amount of text in a single call
- absolutely no memory outside the context
- measured in tokens, not words or characters
- exceeding it causes hard error or quality degradation

> 💡 **Analogy:** Imagine you are given a whiteboard at the start of every conversation. You can write anything on it — the question, the background, previous exchanges, documents you want the model to reference. When the whiteboard is full, you cannot add more. When the conversation ends, the whiteboard is erased completely. The context window is that whiteboard — with a fixed number of squares, each holding one token.


## Counting Tokens in Python with tiktoken

Install tiktoken and count the exact tokens in any text — the same way GPT-4o counts them internally

### Step-by-Step Breakdown

**Load the GPT-4o tokenizer** — tiktoken.encoding_for_model() loads the exact tokenizer OpenAI uses internally for GPT-4o. The encoding is called cl100k_base and is also used by gpt-4 and gpt-3.5-turbo. For Claude and Gemini, use their SDK's token counting methods — but cl100k_base gives a very close approximation for all modern LLMs.

**Encode and count a normal sentence** — enc.encode() returns a list of integer token IDs — one integer per token. The length of that list is the token count. This sentence has 11 words but 12 tokens because punctuation counts separately. The ratio for common English is typically 1.0–1.3 tokens per word.

**Show how rare words cost more tokens** — The rare word 'Antidisestablishmentarianism' splits into 7 tokens despite being one word. This illustrates why technical jargon, medical terms, code identifiers, and non-English text push token counts higher than word counts suggest. When estimating context size, use token counts — never word counts.



In [2]:
!pip install tiktoken

In [3]:
# pip install tiktoken
import tiktoken

# Load the encoder for GPT-4o
enc = tiktoken.encoding_for_model("gpt-4o")

# A plain sentence
text = "The context window is the working memory of the language model."
tokens = enc.encode(text)
print(f"Text   : {text}")
print(f"Tokens : {tokens}")
print(f"Count  : {len(tokens)} tokens  |  {len(text.split())} words")
print(f"Ratio  : {len(tokens)/len(text.split()):.2f} tokens per word\n")

# A rare / long word — splits into many sub-tokens
rare = "Antidisestablishmentarianism is a rare English word."
rare_toks = enc.encode(rare)
print(f"Rare text: {rare}")
print(f"Count    : {len(rare_toks)} tokens  |  {len(rare.split())} words")
print(f"Ratio    : {len(rare_toks)/len(rare.split()):.2f} tokens per word")

Text   : The context window is the working memory of the language model.
Tokens : [976, 3814, 5734, 382, 290, 4113, 8197, 328, 290, 6439, 2359, 13]
Count  : 12 tokens  |  11 words
Ratio  : 1.09 tokens per word

Rare text: Antidisestablishmentarianism is a rare English word.
Count    : 12 tokens  |  6 words
Ratio    : 2.00 tokens per word


## Measuring Your Full Context Size Before Sending

Count the total tokens across system prompt, conversation history, and new message — then calculate what percentage of the context window you are using

### Step-by-Step Breakdown

**Define the context components** — A real API call to GPT-4o has three parts: the system prompt (role definition), the conversation history (all prior turns as a list of role/content dicts), and the new user message. All three count against your context window. In a long conversation, history is almost always the dominant cost — it grows with every exchange.

**Count tokens for each section** — We count tokens separately for each section so we can see exactly where the budget is going. The lambda tok() is a shortcut to avoid repeating enc.encode(). In production code you would add the overhead tokens added by the ChatML format (~4 tokens per message for role tags) but for estimation this gives you an accurate picture.

**Display context usage as a percentage** — Displaying the percentage of the context window used is the key diagnostic. A short conversation like this uses a fraction of a percent of GPT-4o's 128K limit. But in a multi-hour coding session with long assistant replies, you can easily reach 80–90% — and quality problems start well before you hit 100%.



In [4]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")
GPT4O_LIMIT = 128_000
tok = lambda t: len(enc.encode(t))

system = "You are a helpful Python programming tutor. Be concise and clear."
history = [
    {"role": "user",      "content": "What is a list in Python?"},
    {"role": "assistant", "content": "A list is an ordered, mutable sequence: [1, 2, 3]."},
    {"role": "user",      "content": "How do I filter items from a list?"},
    {"role": "assistant", "content": "Use a list comprehension: [x for x in lst if x > 0]."},
]
message = "Now show me how to sort and filter at the same time, with a real example."

sys_tok  = tok(system)
hist_tok = sum(tok(m["content"]) for m in history)
msg_tok  = tok(message)
total    = sys_tok + hist_tok + msg_tok

print(f"System prompt   : {sys_tok:>6,} tokens")
print(f"Chat history    : {hist_tok:>6,} tokens  ({len(history)} messages)")
print(f"New message     : {msg_tok:>6,} tokens")
print(f"Total context   : {total:>6,} tokens  ({total/GPT4O_LIMIT*100:.3f}% of GPT-4o limit)")

System prompt   :     13 tokens
Chat history    :     51 tokens  (4 messages)
New message     :     18 tokens
Total context   :     82 tokens  (0.064% of GPT-4o limit)


## What Happens When Context Gets Too Long?

Filling your context window is not just a capacity problem — it is a quality problem that begins long before you hit the hard limit. The most important finding here comes from Liu et al. at Stanford, published in 2023 under the title 'Lost in the Middle: How Language Models Use Long Contexts.'

### Key Points

- LLMs struggle with information buried in the middle
- retrieval accuracy drops from roughly eighty percent
- quality degrades gradually before you hit the hard limit
- cost scales directly with tokens



## Detecting and Preventing Context Overflow

Build a guard function that checks token count before sending and trims the oldest conversation messages when the context is too large

### Step-by-Step Breakdown

**Count total tokens in a message list** — total_tokens() sums the token count of every message's content. In production you would also add the 4-token overhead per message that OpenAI's ChatML format adds, but for most purposes this gives you an accurate budget estimate within 1-2%.

**Trim oldest messages to fit the budget** — trim_to_fit() enforces a hard budget by dropping the oldest non-system message and repeating until the context fits. The reserve parameter (default 2,000 tokens) holds back space for the model's response so it has room to answer fully. The system message is never dropped — it is the model's instructions and must always be present.

**Test with an oversized context** — The test creates a conversation where two messages contain extremely long repeated strings — simulating a multi-hour coding session with verbose replies. After trim_to_fit() runs, only the messages that fit within the budget remain. In production, call this function just before every API request, ensuring you never hit a context_length_exceeded error.



In [6]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

def total_tokens(messages: list[dict]) -> int:
    return sum(len(enc.encode(m["content"])) for m in messages)

def trim_to_fit(messages: list[dict], limit: int = 128_000, reserve: int = 2_000):
    """Drop oldest non-system messages until the context fits the limit."""
    budget = limit - reserve
    while total_tokens(messages) > budget:
        for i, msg in enumerate(messages):
            if msg["role"] != "system":
                print(f"  Dropped [{msg['role']}]: {msg['content'][:50]}...")
                messages.pop(i)
                break
        else:
            break
    return messages

messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Hello, " + "tell me more. " * 5_000}, # making very long message
    {"role": "assistant", "content": "Sure! " + "Here is info. " * 3_000}, # making very long message
    {"role": "user",      "content": "What is the capital of France?"},
]
safe = trim_to_fit(messages)
print(f"Final: {total_tokens(safe):,} tokens across {len(safe)} messages")

Final: 32,019 tokens across 4 messages


### ✏️ Try It Yourself

Modify the code above — some ideas:
- Change the input values and observe the output
- Add error handling
- Extend the example with one additional feature
